# **Notebook 1a: Data loading: tracking completeness and identifying bad rows**



## 1. Importing the packages

In [1]:
# Loading the packages
from collections import Counter
import pandas as pd
import numpy as np
import warnings
import textwrap
import html
import re

## 2. Loading the dataset

In [2]:
# Loading the data
# Giving a warning when a row has more columns then expected (based of the headings)
# Rows with a warning are not loaded
data = pd.read_csv("dataset2.tsv", sep="\t", on_bad_lines="warn")

/var/folders/c1/_vnm4tbd57174gws_3j9hz7h0000gn/T/ipykernel_32431/383231303.py:4: ParserWarning: Skipping line 53: expected 55 fields, saw 56
Skipping line 111: expected 55 fields, saw 60
Skipping line 243: expected 55 fields, saw 58
Skipping line 297: expected 55 fields, saw 65
Skipping line 301: expected 55 fields, saw 60
Skipping line 420: expected 55 fields, saw 58
Skipping line 443: expected 55 fields, saw 59
Skipping line 536: expected 55 fields, saw 63
Skipping line 582: expected 55 fields, saw 62
Skipping line 631: expected 55 fields, saw 59
Skipping line 672: expected 55 fields, saw 62
Skipping line 797: expected 55 fields, saw 56
Skipping line 924: expected 55 fields, saw 57
Skipping line 948: expected 55 fields, saw 57
Skipping line 955: expected 55 fields, saw 57
Skipping line 958: expected 55 fields, saw 59
Skipping line 1069: expected 55 fields, saw 60
Skipping line 1169: expected 55 fields, saw 57
Skipping line 1170: expected 55 fields, saw 58
Skipping line 1171: expected

### 2a. Calculating the amount of not loaded lines




In [3]:
# Opening the file and counting all rows
with open('dataset2.tsv', 'r') as f:
    total_rows = sum(1 for line in f) - 1  # -1 for header

print(f"Total data rows in file: {total_rows}")

# Counting the amount skipped rows
loaded = len(data) # amount of rows excluded 'bad lines'
skipped = total_rows - loaded

print(f"Amount of loaded rows: {loaded}")
print(f"Amount of skipped rows: {skipped}")
print(f"Percentage of rows that are skipped: {(skipped / total_rows * 100):.2f}%")

Total data rows in file: 26252
Amount of loaded rows: 25917
Amount of skipped rows: 335
Percentage of rows that are skipped: 1.28%


### 2b. Inspecting the extra columns in the 'bad lines'

In [4]:
# Inspecting the 'bad lines'
expected = 55
count_bad_rows = 0
max_to_show = 5  # amount of 'bad lines' to show

with open("dataset2.tsv", "r", encoding="utf-8") as f:
    for line_num, line in enumerate(f, 1):
        fields = line.strip().split("\t")
        if len(fields) > expected:
            count_bad_rows += 1
            if count_bad_rows <= max_to_show:  # Only print if within limit
                main_row = fields[:expected]
                extras = fields[expected:]
                print(f"Line {line_num}")
                for extra in extras:
                    print([extra])
                print()

print("Total rows with extra columns:", count_bad_rows)

Line 53
['Cholerastad is een hoogst eigentijdse, historische roman over de ontregeling van de mens in besmettelijke tijden. Atte Jongstra, die zich voor deze roman liet inspireren door Flauberts grote onvoltooide roman &lt;em&gt;Bouvard en Pécuchet&lt;/em&gt;, verweefde tal van authentieke 19e-eeuwse krantenknipsels in het verhaal.']

Line 111
['werken met het menu Start, bureaublad en de taakbalk&lt;br&gt;&lt;br&gt;\\n•']
['kennismaken met nieuwe en vernieuwde programma’s&lt;br&gt;&lt;br&gt;\\n•']
['werken met foto, video en muziek&lt;br&gt;&lt;br&gt;\\n•']
['diverse instellingen bekijken en wijzigen&lt;br&gt;&lt;br&gt;\\n•']
['beveiliging van de pc&lt;br&gt;&lt;br&gt;&lt;br&gt;\\n&lt;b&gt;Geschikt voor:&lt;/b&gt;&lt;br&gt;\\nWindows 11 op een desktop-pc of laptop&lt;br&gt;&lt;br&gt;&lt;br&gt;\\n&lt;b&gt;Benodigde voorkennis:&lt;/b&gt;&lt;br&gt;\\nU kunt al werken met een eerdere versie van Windows, zoals Windows 10, of een tablet of smartphone.\\n']

Line 243
['&lt;li&gt;Inclusief gr

## 3. Renaming and adding columns for easier interpretation

In [5]:
# Renaming three columns

# This is the AI summary
data.rename(columns={"first_version": "automatic_output"}, inplace=True)

# This is the corrected summary
# If this is EMPTY: only one human corrected the AI summary, this correction is in final_edit
# If this is NOT EMPTY: two humans corrected the AI summary, the correction by human 1 is in intermediate_edit while the correction by human 2 is in final_edit
data.rename(columns={"second_version": "intermediate_edit"}, inplace=True)

# This is always the final edit
data.rename(columns={"annotated_text": "final_edit"}, inplace=True)

## 4. Looking into signs in the text

### 4a. Inspecting literal escape text

In [6]:
# Checking for literal escape text (backslash + letter as visible characters, e.g. \n, \t)
pattern_to_check = [
    (r"\\[tnrvfu]", True, "literal escape text (\\\\t, \\\\n, \\\\r, etc.)"),]

for pattern, regex, label in pattern_to_check:
    for col in data.columns:
        mask = data[col].astype(str).str.contains(pattern, regex=regex, na=False)
        count = mask.sum()
        if count > 0:
            print(f"\n{label} in '{col}': {count} rows")
# Example
print("\n---------------------------------------------Example:---------------------------------------------")
matches = data.loc[mask].astype(str).agg(' | '.join, axis=1)
for i in range(1):
    print(textwrap.fill(matches.iloc[i], width=100))
    print()  # lege regel ertussen voor leesbaarheid


literal escape text (\\t, \\n, \\r, etc.) in 'final_edit': 23326 rows

literal escape text (\\t, \\n, \\r, etc.) in 'automatic_output': 17505 rows

literal escape text (\\t, \\n, \\r, etc.) in 'intermediate_edit': 18987 rows

literal escape text (\\t, \\n, \\r, etc.) in 'awards': 7 rows

literal escape text (\\t, \\n, \\r, etc.) in 'countries_published': 19 rows

literal escape text (\\t, \\n, \\r, etc.) in 'known_works': 3 rows

literal escape text (\\t, \\n, \\r, etc.) in 'blurb': 19356 rows

---------------------------------------------Example:---------------------------------------------
9789461230485 | mas_baf_processed_7 | Een reisgids voor Moravië, gelegen in het oosten van Tsjechië.
Moravië is overwegend landelijk van karakter met een paar grote steden, en is onder andere bekend om
zijn wijnen. Moravië wordt overzichtelijk in twee regio’s beschreven, met overzichtskaarten,
kleurenfoto’s, achtergrondinformatie en tips voor horeca en overnachtingen. Helder en prettig
geschreven.

In [7]:
# Deleting literal escape text (\t, \n, \r, etc. as visible backslash + letter)
text_cols = data.select_dtypes(include="object").columns
for col in text_cols:
    data[col] = data[col].astype(str)
    # Replacing literal escape text with a space
    data[col] = data[col].str.replace(r"\\[tnrvfu]", " ", regex=True)
    # Collapsing multiple spaces into one and strip leading/trailing whitespace
    data[col] = data[col].str.replace(r" +", " ", regex=True).str.strip()

### 4b. Checking for HTML tags and entities

In [8]:
# Checking for HTML tags and entities
def check_html_tags_and_entities(data, text_cols):
    patterns = [
        (r'<[^>]*>',         "HTML tags (e.g., <div>, <p>)"),
        (r'&[a-zA-Z0-9#]+;', "HTML entities (e.g., &amp;, &nbsp;)"),
    ]

    example = None

    for pattern, label in patterns:
        for col in text_cols:
            mask = data[col].astype(str).str.contains(pattern, regex=True, na=False)
            count = mask.sum()

            if count > 0:
                print(f"{label} in '{col}': {count} rows")

check_html_tags_and_entities(data, text_cols)

HTML tags (e.g., <div>, <p>) in 'automatic_output': 50 rows
HTML tags (e.g., <div>, <p>) in 'pseudonym': 1 rows
HTML tags (e.g., <div>, <p>) in 'bibliography': 33 rows
HTML tags (e.g., <div>, <p>) in 'debut': 7 rows
HTML tags (e.g., <div>, <p>) in 'blurb': 46 rows
HTML entities (e.g., &amp;, &nbsp;) in 'final_edit': 2 rows
HTML entities (e.g., &amp;, &nbsp;) in 'intermediate_edit': 1 rows
HTML entities (e.g., &amp;, &nbsp;) in 'countries_published': 5 rows
HTML entities (e.g., &amp;, &nbsp;) in 'blurb': 6767 rows


In [9]:
# Look into a few random exmples of /em (html tag) in the data
target_terms = ['/em']

for term in target_terms:
    print(f"\n=== '{term}' in context (3 examples) ===")

    # Create a boolean mask for rows that contain the term
    mask = data['blurb'].astype(str).str.contains(term, regex=False, na=False)

    if mask.any():
        # Take up to 3 random examples from matching rows
        sample = data.loc[mask, 'blurb'].sample(min(3, mask.sum()), random_state=42)

        for val in sample:
            # Convert value to string (safety step)
            text = str(val)

            # Find position of the term in the text
            pos = text.find(term)

            # Extract a small snippet around the term
            snippet = text[max(0, pos-30):pos+len(term)+30].replace("\n", " ")

            # Print the snippet with padding for readability
            print(f"  ...{snippet}...")

  # Note: look sometimes < and > signs are written as &lt; and &gt; (in entities) so instead of deleting them
  # We should first change them back to < so the part in between can be detected to be deleted


=== '/em' in context (3 examples) ===
  ...9;&lt;br /&gt;&lt;br /&gt;&lt;/em&gt;Na het overlijden van haar...
  ...em&gt;De grote hummelheisa&lt;/em&gt; is een deel in de populai...
  ...&gt;Moeders des Vaderlands&lt;/em&gt; toont deze veelal vergete...


In [10]:
# Deleting html tags and entities
def remove_html_tags_and_entities(text):
    if pd.isna(text):
        return text
    text = str(text)

    # STAP 1: Decodeer HTML entities EERST
    # Dit zet &lt;em&gt; om naar <em>, &amp; naar &, etc.
    text = html.unescape(text)

    # STAP 2: Nu pas de echte HTML-tags verwijderen
    # &lt;em&gt; is nu <em> en wordt herkend
    clean = re.sub(r'<[^>]*>', ' ', text)

    # STAP 3: Eventueel nog overgebleven losse entities (zeldzaam na unescape)
    clean = re.sub(r'&[a-zA-Z0-9#]+;', ' ', clean)

    # Whitespace opruimen
    clean = re.sub(r'\s+', ' ', clean).strip()

    return clean


# Defining text_cols to include all object type columns
text_cols = data.select_dtypes(include="object").columns
# Apply the cleaning function to all text columns
for col in text_cols:
    data[col] = data[col].apply(remove_html_tags_and_entities)

print("HTML signs deleted from text columns.")

HTML signs deleted from text columns.


### 4c. Checking what signs are left

In [11]:
counts = Counter()
for col in text_cols:
    for value in data[col].dropna().astype(str):
        # Single weird characters not in allowed set
        counts.update(re.findall(r"[^\w\s\'\-\.\,\!\?\;\:\(\)\"\/\&]", value))
        # HTML entities (letter-based AND numeric)
        counts.update(re.findall(r"&[a-zA-Z0-9#]+;", value))
        # Slash-prefixed sequences (escape text or HTML tag fragments)
        counts.update(re.findall(r"[\\\/]\w+", value))

for char, count in counts.most_common(100):
    print(f"{repr(char):<15} {count}")

'|'             415749
'’'             63641
'‘'             35638
'–'             9640
'['             9434
']'             9363
'̈'             5533
'…'             5305
'́'             4226
'*'             3509
'—'             2807
'+'             1188
'\xad'          1012
'“'             924
'•'             907
'”'             894
'®'             869
'̀'             796
'='             790
'·'             767
'̂'             736
'�'             480
'»'             450
'#'             446
'}'             444
'{'             423
'@'             414
'€'             405
'̄'             400
'>'             364
'→'             351
'̌'             338
'/subgenre'     315
'̧'             309
'/het'          303
'/m'            285
'%'             259
'/verhaal'      259
'«'             258
'̃'             252
'̆'             247
'/of'           234
'©'             154
'/vrouw'        145
'\u200b'        138
'͏'             113
'/Infodok'      109
'̊'             103
'™'             101
'/w

In [12]:
# Delete \xad and \u200b by hand

# Defining text_cols to include all object type columns
text_cols = data.select_dtypes(include="object").columns

# Apply replacements to all text columns
for col in text_cols:
    data[col] = data[col].apply(
        lambda text: text if pd.isna(text) else str(text).replace("\u200b", "").replace("\xad", ""))

print("Invisible characters removed from text columns.")

Invisible characters removed from text columns.


### 4d. Looking into alonestanding letters

Alonestanding letters could be the result of issues in how the data was captured, for exampel z'n could be captured as z n. It could also be another issue to look into that we looked into blurbs with 2 or more, 3 or more and 4 or more alonestanding letters after each other to see where the real problems lay.

In [13]:
# Quick comparison: spaced-letter sequences in blurb vs final_edit
seq_pattern = re.compile(r"(?:\b[a-zA-Z]\s+){2,}\b[a-zA-Z]\b")

print("=== Spaced-letter sequences (3+) per column ===\n")
for col in ["blurb", "final_edit"]:
    series = data[col].dropna()
    rows_affected = series.apply(lambda t: bool(seq_pattern.search(t))).sum()
    total_matches = series.apply(lambda t: len(seq_pattern.findall(t))).sum()
    n_total = len(series)

    print(f"{col}:")
    print(f"  Rows affected: {rows_affected:,} / {n_total:,} ({rows_affected/n_total*100:.2f}%)")
    print(f"  Total matches: {total_matches:,}\n")

=== Spaced-letter sequences (3+) per column ===

blurb:
  Rows affected: 103 / 25,917 (0.40%)
  Total matches: 252

final_edit:
  Rows affected: 1 / 25,917 (0.00%)
  Total matches: 1



In [14]:
# Getting insight in alonestanding letters in the blurb
# Testing different amount of alonestanding letters togheter
patterns = {
    "4+ letters":  re.compile(r"(?:\b[a-zA-Z]\s+){3,}\b[a-zA-Z]\b"),
    "3+ letters":  re.compile(r"(?:\b[a-zA-Z]\s+){2,}\b[a-zA-Z]\b"),
    "2+ letters":  re.compile(r"(?:\b[a-zA-Z]\s+){1,}\b[a-zA-Z]\b"),
}
print("=== Comparison ===\n")
for label, pat in patterns.items():
    total_matches = 0
    rows_affected = 0
    for text in data["blurb"].dropna():
        matches = pat.findall(text)
        if matches:
            rows_affected += 1
            total_matches += len(matches)
    print(f"{label}:")
    print(f"  Rows affected: {rows_affected:,}")
    print(f"  Total matches: {total_matches:,}\n")
# Inspecting the matches of every levels
def show_examples(pattern, label, n_examples=20, exact_letters=None):
    print(f"\n=== Voorbeelden: {label} ===\n")
    seen = 0
    for text in data["blurb"].dropna():
        matches = pattern.findall(text)
        for m in matches:
            n_letters = len(m.replace(" ", ""))
            # Filtering: only show matches with exact the amount of letters (no overlap)
            if exact_letters is not None and n_letters != exact_letters:
                continue
            idx = text.find(m)
            if idx >= 0:
                start = max(0, idx - 40)
                end   = min(len(text), idx + len(m) + 40)
                print(f"  [{n_letters} letters] ...{text[start:end]}...")
                seen += 1
                if seen >= n_examples:
                    return
# Showing examples
show_examples(patterns["2+ letters"], "exact 2 letters", exact_letters=2)
show_examples(patterns["3+ letters"], "exact 3 letters", exact_letters=3)
show_examples(patterns["4+ letters"], "4+ letters")


# Inspection revealed two recurring noise patterns in blurbs:
# (1) decorative spaced-out title text (e.g., "f a v o r i e t e n"), and
# (2) What seem to be extraction artefacts producing isolated single letters with whitespace between them.
# Based on this exploration, sequences of three or more space-separated single letters are removed.

=== Comparison ===

4+ letters:
  Rows affected: 77
  Total matches: 188

3+ letters:
  Rows affected: 103
  Total matches: 252

2+ letters:
  Rows affected: 252
  Total matches: 498


=== Voorbeelden: exact 2 letters ===

  [2 letters] ... Maar, zoals Leonard Cohen zingt: there’s a crack in everything, that’s how the lig...
  [2 letters] ...geten en gelukzalig in de zetel zit met h r huisdraakje op schoot, ontploft ze, en ...
  [2 letters] ...H t hoopvolle boek voor iedereen die verlan...
  [2 letters] ...dwijd succes. Met deze bundel kunnen nu n g meer lezers kennismaken met de heldhaft...
  [2 letters] ...t ook precies wat ze wel en niet wilde. H p, naar het opvoedkamp van Jeroen Oomen. ...
  [2 letters] ...aan anderen te denken. Ze zijn, kortom, h l vermoeiend. Aan de hand van hetzelfde v...
  [2 letters] ...ind negentiende eeuw legde Santiago Ram n y Cajal de basis voor de moderne neurowet...
  [2 letters] ...ind negentiende eeuw legde Santiago Ram n y Cajal de basis voor de moder

In [15]:
# Deleting alonestanding letters where there are 3 or more after each other

# Pattern: 3+ space-separated single letters (lowercase or uppercase)
spaced_letter_seq = re.compile(r"(?:\b[a-zA-Z]\s+){2,}\b[a-zA-Z]\b")

def clean_spaced_letters(text):
    if not isinstance(text, str):
        return text
    # Replacing sequences with a single space to prevent word merging
    cleaned = spaced_letter_seq.sub(" ", text)
    # Collapsing any resulting multiple spaces into one
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned

# Applying to blurb (and optionally to final_edit for consistency)
data["blurb"] = data["blurb"].apply(clean_spaced_letters)

print("Spaced-letter sequences (3+) removed from blurb and final_edit.")

Spaced-letter sequences (3+) removed from blurb and final_edit.


In [16]:
zoekterm = "Drie minuten van een van"

# Vind alle rijen waar de blurb deze tekst bevat
gevonden = data[data['blurb'].str.contains(zoekterm, na=False, regex=False)]

print(f"Aantal matches: {len(gevonden)}")

# Toon de gevonden blurbs
pd.set_option("display.max_colwidth", None)
for idx, row in gevonden.iterrows():
    print(f"\n--- Row {idx} ---")
    print(row["blurb"])

print(gevonden)

Aantal matches: 1

--- Row 8295 ---
G Y UL A POZ Drie minuten van een van tevoren al verloren luchtgevecht ... Een miniem detail gedurende de Tweede Wereldoorlog ... Het is niet meer dan een kort treffen, kort maar heftig, tussen een Hongaarse jachtbommenwerper Me 210 Ca-1 en negen Yak-9 Sovjetjagers. Het is geen avonturenroman, maar het waargebeurde verhaal van de grootvader van de auteur, die hem de liefde voor tekenen heeft bijgebracht. Een verhaal gebaseerd op het dagboek dat de gevangen vliegenier na de oorlog bijhield. THE OLD TIGER bitis halálát okoztau Hazzetkezve már fudd ibutette war, hou lad awiki ket tal 9"-ct guagott, legalem! Frecher ellenségnek . 17./ 1944. 23 oktober. - 1h-06 Vlieghoogte: 2300 m. "Bombardement met duikbommenwerpers" "Z-094" Piloot: V. Orbán József. Observator: sgt Tibor Vörös. Missie: bombardement met duikbommenwerpers op Sovjet-tanks die worden uitgeladen op het station van Orosháza Sterke verdediging: zwaar en licht afweergeschut. 9 .Russische jagers 

## 5. Final data after cleaning


In [17]:
print("First 10 rows of 'automatic_output':\n", data['automatic_output'].head(10))
print("\n First 10 rows of 'intermediate_edit': \n", data['intermediate_edit'].head(10))
print("\n First 10 rows of 'final_edit': \n", data['final_edit'].head(10))

First 10 rows of 'automatic_output':
 0                                                                                                                                                                                                                                                                                                                                                                    Een boek over het leven en de adel. Nuchter en toegankelijk geschreven. Voor een brede lezersgroep. Met volop praktische tips. Albert Gielen (1957) is een Nederlandse auteur. Het boek maakt deel uit van de serie: 'Odyssee Reisgidsen'.
1    ‘De rivier’ is een historische roman over drie Finse kinderen die aan het begin van de 20e eeuw vluchten naar frontierstaat Washington. Daar voegen ze zich bij een Finse houtkapgemeenschap, die snel in ontwikkeling is en waar een radicale arbeidersbeweging wordt opgericht. Broers Ilmari en Matti worden geconfronteerd met de opwinding en het gevaar van de wilderni

## 6. Looking into NaN values in the data

In [18]:
# Replace string 'nan' with actual NaN values in the relevant columns
for col in ["intermediate_edit", "final_edit", "automatic_output"]:
    data[col] = data[col].replace('nan', np.nan)

## 7. Saving the data as csv

In [19]:
data.to_csv("dataset_cleaned.tsv", sep="\t", index=False)